In [1]:
%pip install -q transformers datasets accelerate peft sacrebleu sentencepiece
import os
import random
import json
import time

import numpy as np
import pandas as pd
import torch
import transformers
import peft
import sacrebleu

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle dataset location
INPUT_DIR = (
    "/kaggle/input/datasets/dishita05/"
    "gen-ai-ia3-language-translation-dataset"
)

# Separate outputs for the LLaMA experiment
WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
DATA_DIR = os.path.join(WORK_DIR, "data2")
MODEL_DIR = os.path.join(WORK_DIR, "models")
RESULTS_DIR = os.path.join(WORK_DIR, "results")

LLAMA_MODEL_DIR = os.path.join(MODEL_DIR, "llama_lora")
LLAMA_RESULTS_DIR = os.path.join(RESULTS_DIR, "llama")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(LLAMA_MODEL_DIR, exist_ok=True)
os.makedirs(LLAMA_RESULTS_DIR, exist_ok=True)

# Load the unchanged dataset
train_df = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(INPUT_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    )

print("\nDataset sizes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

assert list(train_df.columns) == ["en", "es"]
assert list(val_df.columns) == ["en", "es"]
assert list(test_df.columns) == ["en", "es"]
assert len(train_df) == 20000
assert len(val_df) == 1000
assert len(test_df) == 1000

print("\nEnvironment and dataset checks passed.")

Note: you may need to restart the kernel to use updated packages.
Transformers: 5.16.1
PEFT: 0.20.0
PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4
GPU memory (GB): 15.64

Dataset sizes:
Train: (20000, 2)
Validation: (1000, 2)
Test: (1000, 2)

Environment and dataset checks passed.


In [2]:
# TinyLlama configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MAX_LENGTH = 256
MAX_NEW_TOKENS = 96

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

LLAMA_OUTPUT_DIR = os.path.join(
    MODEL_DIR, "llama_lora"
)
LLAMA_RESULTS_DIR = os.path.join(
    RESULTS_DIR, "llama"
)

os.makedirs(LLAMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(LLAMA_RESULTS_DIR, exist_ok=True)

print("Model:", MODEL_NAME)
print("Train/validation/test:", len(train_df), len(val_df), len(test_df))
print("Effective batch size:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("GPU:", torch.cuda.get_device_name(0))

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Train/validation/test: 20000 1000 1000
Effective batch size: 16
GPU: Tesla T4


In [3]:
%pip install -q bitsandbytes
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Note: you may need to restart the kernel to use updated packages.
bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4


In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Configure 4-bit quantization for the T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading TinyLlama tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading TinyLlama model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("\nModel loaded successfully.")
print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Device:", model.device)
print(
    "GPU memory allocated (GB):",
    round(torch.cuda.memory_allocated() / 1e9, 2)
)

Loading TinyLlama tokenizer...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model loaded successfully.
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Parameters: 615,606,272
Device: cuda:0
GPU memory allocated (GB): 0.23


In [5]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# Prepare the quantized model for parameter-efficient training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print(
    "\nGPU memory allocated (GB):",
    round(torch.cuda.memory_allocated() / 1e9, 2)
)

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044

GPU memory allocated (GB): 0.36


In [6]:

from datasets import Dataset

def preprocess_translation(batch):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for en, es in zip(batch["en"], batch["es"]):
        prompt = (
            "Translate the following English text into Spanish.\n"
            f"English: {en}\n"
            "Spanish:"
        )

        # Tokenize prompt and Spanish target separately.
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=False
        )["input_ids"]

        target_ids = tokenizer(
            " " + str(es),
            add_special_tokens=False
        )["input_ids"]

        # Explicitly terminate the target with EOS.
        target_ids = target_ids + [tokenizer.eos_token_id]

        input_ids = (prompt_ids + target_ids)[:MAX_LENGTH]

        # Ignore prompt tokens in the loss; train on Spanish tokens only.
        labels = (
            [-100] * len(prompt_ids) + target_ids
        )[:MAX_LENGTH]

        attention_mask = [1] * len(input_ids)

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels
    }


print("Preparing masked training examples...")

train_tokenized = Dataset.from_pandas(
    train_df[["en", "es"]].reset_index(drop=True)
).map(
    preprocess_translation,
    batched=True,
    remove_columns=["en", "es"],
    desc="Preparing training set"
)

val_tokenized = Dataset.from_pandas(
    val_df[["en", "es"]].reset_index(drop=True)
).map(
    preprocess_translation,
    batched=True,
    remove_columns=["en", "es"],
    desc="Preparing validation set"
)

print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))

sample = train_tokenized[0]
print("Total tokens:", len(sample["input_ids"]))
print("Prompt tokens masked:", sum(x == -100 for x in sample["labels"]))
print("Spanish target tokens:", sum(x != -100 for x in sample["labels"]))


Preparing masked training examples...


Preparing training set:   0%|          | 0/20000 [00:00<?, ? examples/s]

Preparing validation set:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training examples: 20000
Validation examples: 1000
Total tokens: 51
Prompt tokens masked: 31
Spanish target tokens: 20


In [7]:

import torch

def collate_translation_batch(features):
    batch_size = len(features)
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = torch.full(
        (batch_size, max_len),
        tokenizer.pad_token_id,
        dtype=torch.long
    )
    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )
    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    for i, feature in enumerate(features):
        length = len(feature["input_ids"])

        input_ids[i, :length] = torch.tensor(
            feature["input_ids"], dtype=torch.long
        )
        attention_mask[i, :length] = torch.tensor(
            feature["attention_mask"], dtype=torch.long
        )
        labels[i, :length] = torch.tensor(
            feature["labels"], dtype=torch.long
        )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


# Verify the collator before training.
test_batch = collate_translation_batch(
    [train_tokenized[0], train_tokenized[1]]
)

print("Input shape:", test_batch["input_ids"].shape)
print("Attention mask shape:", test_batch["attention_mask"].shape)
print("Labels shape:", test_batch["labels"].shape)
print("Masked prompt labels:", (test_batch["labels"] == -100).sum().item())
print("Collator verification successful.")


Input shape: torch.Size([2, 122])
Attention mask shape: torch.Size([2, 122])
Labels shape: torch.Size([2, 122])
Masked prompt labels: 161
Collator verification successful.


In [8]:

import inspect
import transformers
from transformers import TrainingArguments, Trainer

print("Transformers version:", transformers.__version__)

training_args_signature = inspect.signature(TrainingArguments.__init__)
trainer_signature = inspect.signature(Trainer.__init__)

print("\nTrainingArguments supports:")
for name in [
    "eval_strategy",
    "evaluation_strategy",
    "save_strategy",
    "logging_steps",
    "num_train_epochs",
    "gradient_accumulation_steps",
    "gradient_checkpointing",
    "fp16",
    "optim",
    "report_to",
    "remove_unused_columns",
    "save_total_limit"
]:
    print(f"{name}: {name in training_args_signature.parameters}")

print("\nTrainer supports:")
for name in ["processing_class", "tokenizer", "data_collator"]:
    print(f"{name}: {name in trainer_signature.parameters}")


Transformers version: 5.16.1

TrainingArguments supports:
eval_strategy: True
evaluation_strategy: False
save_strategy: True
logging_steps: True
num_train_epochs: True
gradient_accumulation_steps: True
gradient_checkpointing: True
fp16: True
optim: True
report_to: True
remove_unused_columns: True
save_total_limit: True

Trainer supports:
processing_class: True
tokenizer: False
data_collator: True


In [9]:

import os
import json
import shutil
from datetime import datetime

# Create a dedicated checkpoint directory.
BACKUP_DIR = "/kaggle/working/tinyllama_recovery"
os.makedirs(BACKUP_DIR, exist_ok=True)

print("Recovery directory:", BACKUP_DIR)
print("Directory ready:", os.path.isdir(BACKUP_DIR))
print("Model output:", LLAMA_OUTPUT_DIR)
print("Results output:", LLAMA_RESULTS_DIR)


Recovery directory: /kaggle/working/tinyllama_recovery
Directory ready: True
Model output: /kaggle/working/ICT4403_MT_Project/models/llama_lora
Results output: /kaggle/working/ICT4403_MT_Project/results/llama


In [10]:
import os
import json
import time
import inspect
import torch
from transformers import TrainingArguments, Trainer

os.makedirs(LLAMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(LLAMA_RESULTS_DIR, exist_ok=True)

# Configure training arguments compatible with this Transformers version.
args_dict = {
    "output_dir": LLAMA_OUTPUT_DIR,
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "fp16": True,
    "gradient_checkpointing": True,
    "optim": "paged_adamw_8bit",
    "report_to": "none",
    "remove_unused_columns": False,
    "dataloader_pin_memory": False,
}

# Filter arguments defensively against the installed API.
supported_args = inspect.signature(
    TrainingArguments.__init__
).parameters

args_dict = {
    key: value for key, value in args_dict.items()
    if key in supported_args
}

training_args = TrainingArguments(**args_dict)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=collate_translation_batch,
    processing_class=tokenizer,
)

print("Training configuration ready.")
print("Epochs:", NUM_EPOCHS)
print("Training rows:", len(train_tokenized))
print("Validation rows:", len(val_tokenized))
print("Trainable parameters:", model.get_nb_trainable_parameters()[0])
print("Total parameters:", model.get_nb_trainable_parameters()[1])
print("Starting fine-tuning...")

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

train_result = trainer.train()

training_seconds = time.time() - start_time
peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

# Save the trained LoRA adapter and tokenizer.
trainer.save_model(LLAMA_OUTPUT_DIR)
tokenizer.save_pretrained(LLAMA_OUTPUT_DIR)


# Create a portable archive of the trained adapter.
adapter_archive = shutil.make_archive(
    "/kaggle/working/tinyllama_lora_adapter",
    "zip",
    root_dir=LLAMA_OUTPUT_DIR
)

print("Adapter archive created:", adapter_archive)
print("Archive size (MB):", round(os.path.getsize(adapter_archive) / (1024**2), 2))

# Copy training summary into the recovery directory.
summary_source = os.path.join(
    LLAMA_RESULTS_DIR, "llama_training_summary.json"
)
if os.path.exists(summary_source):
    shutil.copy2(summary_source, BACKUP_DIR)

print("Training backup checkpoint complete.")


# Record the actual training output.
training_summary = {
    "model": MODEL_NAME,
    "method": "4-bit QLoRA",
    "epochs": NUM_EPOCHS,
    "training_examples": len(train_tokenized),
    "validation_examples": len(val_tokenized),
    "training_time_seconds": training_seconds,
    "training_time_minutes": training_seconds / 60,
    "peak_gpu_memory_gb": peak_gpu_memory_gb,
    "trainable_parameters": model.get_nb_trainable_parameters()[0],
    "total_parameters": model.get_nb_trainable_parameters()[1],
    "train_metrics": train_result.metrics,
}

with open(
    os.path.join(LLAMA_RESULTS_DIR, "llama_training_summary.json"),
    "w"
) as f:
    json.dump(training_summary, f, indent=2, default=str)

print("\nFine-tuning finished.")
print(f"Training time: {training_seconds / 60:.2f} minutes")
print(f"Peak GPU memory: {peak_gpu_memory_gb:.2f} GB")
print("Adapter saved to:", LLAMA_OUTPUT_DIR)
print("Training summary saved to:", LLAMA_RESULTS_DIR)
print("Training metrics:", train_result.metrics)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Training configuration ready.
Epochs: 1
Training rows: 20000
Validation rows: 1000
Trainable parameters: 2252800
Total parameters: 1102301184
Starting fine-tuning...


Epoch,Training Loss,Validation Loss
1,1.334749,1.311874


Adapter archive created: /kaggle/working/tinyllama_lora_adapter.zip
Archive size (MB): 21.38
Training backup checkpoint complete.

Fine-tuning finished.
Training time: 62.96 minutes
Peak GPU memory: 0.48 GB
Adapter saved to: /kaggle/working/ICT4403_MT_Project/models/llama_lora
Training summary saved to: /kaggle/working/ICT4403_MT_Project/results/llama
Training metrics: {'train_runtime': 3777.1757, 'train_samples_per_second': 5.295, 'train_steps_per_second': 0.331, 'total_flos': 8978502611140608.0, 'train_loss': 1.339920098876953, 'epoch': 1.0}


In [11]:

import os
import time
import json
import pandas as pd
import torch

model.eval()
model.config.use_cache = True
tokenizer.padding_side = "left"

# Use the same test set as the other experiments.
test_inputs = test_df["en"].astype(str).tolist()
test_references = test_df["es"].astype(str).tolist()

predictions = []
batch_size = EVAL_BATCH_SIZE

print("Generating TinyLlama translations...")
start_time = time.time()

for start in range(0, len(test_inputs), batch_size):
    batch_texts = test_inputs[start:start + batch_size]

    prompts = [
        "Translate the following English text into Spanish.\n"
        f"English: {text}\n"
        "Spanish:"
        for text in batch_texts
    ]

    encoded = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

    # Move inputs to the model's input device.
    input_device = model.get_input_embeddings().weight.device
    encoded = {k: v.to(input_device) for k, v in encoded.items()}

    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    # Decoder-only models return prompt + generated continuation.
    prompt_width = encoded["input_ids"].shape[1]
    new_tokens = generated[:, prompt_width:]

    batch_predictions = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True
    )

    predictions.extend(
        [p.strip() for p in batch_predictions]
    )

    if (start + len(batch_texts)) % 100 == 0:
        print(f"Processed {min(start + len(batch_texts), len(test_inputs))}/1000")

inference_seconds = time.time() - start_time

prediction_df = pd.DataFrame({
    "en": test_inputs,
    "reference": test_references,
    "prediction": predictions
})

prediction_path = os.path.join(
    LLAMA_RESULTS_DIR,
    "llama_predictions.csv"
)
prediction_df.to_csv(prediction_path, index=False)

inference_summary = {
    "test_examples": len(predictions),
    "inference_time_seconds": inference_seconds,
    "seconds_per_example": inference_seconds / len(predictions),
    "decoding": "greedy",
    "max_new_tokens": MAX_NEW_TOKENS,
    "predictions_path": prediction_path
}

with open(
    os.path.join(LLAMA_RESULTS_DIR, "llama_inference_summary.json"),
    "w"
) as f:
    json.dump(inference_summary, f, indent=2)

print("\nInference complete.")
print(f"Total time: {inference_seconds:.2f} seconds")
print(f"Average time/example: {inference_seconds / len(predictions):.4f} seconds")
print("Predictions saved to:", prediction_path)

print("\nSample predictions:")
display(prediction_df.head(10))


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating TinyLlama translations...


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 100/1000


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

...

[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 900/1000


[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 1000/1000

Inference complete.
Total time: 1150.44 seconds
Average time/example: 1.1504 seconds
Predictions saved to: /kaggle/working/ICT4403_MT_Project/results/llama/llama_predictions.csv

Sample predictions:


,en,reference,prediction
0,We humans will inevitably end up controlling o...,Nosotros humanos terminaremos inevitablemente ...,Nosotros seremos inevitablemente los que contr...
1,If she had not rented quarter in this building...,Si no hubiera rentado un cuarto en este edific...,Si no hubiera comprado cuarto en este edificio...
2,I know what the lies were.,Sé cuáles eran las mentiras.,Siento lo que fueron las mentiras.
3,- Here's your buddy for a party...,! - A eso a venido la chavala a la fiesta...,- Aquí te trae tu amigo para una fiesta...
4,This is not possible for a man of flesh and bl...,Esto no es posible para un hombre de carne y h...,No es posible para un hombre de sangre y carne.
5,"Yes, you must be Reverend Smallbone.",- Sí. Usted debe ser el reverendo Smallbone.,"Sí, debes ser Reverendito Smallbone."
6,"If diesel fuel or fuel-oil is used, at minimum...","En caso de uso de gasóleo o fuel-oil, necesita...",Si se usa el combustible de gasolina o el acei...
7,I asked what you are.,Te pregunto qué eres.,Me pregunté qué eres.
8,- Go delicate?,- ¿Ir con cuidado?,- ¿Vamos a ser delicados?
9,He adds that one senator resigned in protest a...,Añade que un senador dimitió para manifestar s...,El senador añade que uno de los senadores renu...


In [12]:

import os
import json
import sacrebleu

# Use the saved predictions to ensure the evaluated data is exactly
# the same data that was generated during inference.
pred_path = os.path.join(
    LLAMA_RESULTS_DIR, "llama_predictions.csv"
)
pred_df = pd.read_csv(pred_path)

assert len(pred_df) == 1000, "Expected 1,000 test predictions."
assert pred_df["prediction"].notna().all()
assert pred_df["reference"].notna().all()

predictions = pred_df["prediction"].astype(str).tolist()
references = pred_df["reference"].astype(str).tolist()

bleu = sacrebleu.corpus_bleu(predictions, [references])
chrf = sacrebleu.corpus_chrf(predictions, [references])

metrics = {
    "model": MODEL_NAME,
    "method": "4-bit QLoRA",
    "test_examples": len(predictions),
    "bleu": bleu.score,
    "chrf": chrf.score,
    "training_time_minutes": 63.43,
    "inference_time_seconds": 1169.79,
    "seconds_per_example": 1169.79 / len(predictions),
    "trainable_parameters": 2252800,
    "total_parameters": 1102301184,
    "training_loss": 1.3399006774902344,
    "validation_loss": 1.311869
}

metrics_path = os.path.join(
    LLAMA_RESULTS_DIR, "llama_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("TinyLlama evaluation results")
print("-" * 35)
print(f"BLEU: {bleu.score:.4f}")
print(f"chrF: {chrf.score:.4f}")
print(f"Test examples: {len(predictions)}")
print(f"Inference time: {1169.79:.2f} seconds")
print("\nMetrics saved to:", metrics_path)


TinyLlama evaluation results
-----------------------------------
BLEU: 23.8893
chrF: 48.2915
Test examples: 1000
Inference time: 1169.79 seconds

Metrics saved to: /kaggle/working/ICT4403_MT_Project/results/llama/llama_metrics.json


In [17]:

import os
import json
import shutil
import zipfile

WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
RESULTS_DIR = os.path.join(WORK_DIR, "results", "llama")
BACKUP_DIR = "/kaggle/working/tinyllama_recovery"

os.makedirs(BACKUP_DIR, exist_ok=True)

# Correct the metrics to match the reported inference run.
metrics_path = os.path.join(RESULTS_DIR, "llama_metrics.json")

with open(metrics_path, "r") as f:
    metrics = json.load(f)

metrics["inference_time_seconds"] = 1150.44
metrics["seconds_per_example"] = 1150.44 / 1000

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

# Copy predictions and metrics into the recovery folder.
for filename in [
    "llama_predictions.csv",
    "llama_metrics.json",
    "llama_training_summary.json",
    "llama_inference_summary.json"
]:
    source = os.path.join(RESULTS_DIR, filename)
    if os.path.exists(source):
        shutil.copy2(source, BACKUP_DIR)
        print("Backed up:", filename)
    else:
        print("Not found:", filename)

# Bundle available results for download.
bundle_path = "/kaggle/working/tinyllama_results_backup.zip"

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(BACKUP_DIR):
        for filename in files:
            path = os.path.join(root, filename)
            zf.write(path, arcname=os.path.relpath(path, BACKUP_DIR))

print("\nResults archive:", bundle_path)
print("Archive size (MB):", round(os.path.getsize(bundle_path) / (1024**2), 2))


Backed up: llama_predictions.csv
Backed up: llama_metrics.json
Backed up: llama_training_summary.json
Backed up: llama_inference_summary.json

Results archive: /kaggle/working/tinyllama_results_backup.zip
Archive size (MB): 0.09


In [18]:

from IPython.display import display, FileLink
import os
import json
import shutil
import zipfile

WORK_DIR = "/kaggle/working/ICT4403_MT_Project"
RESULTS_DIR = os.path.join(WORK_DIR, "results", "llama")
MODEL_DIR = os.path.join(WORK_DIR, "models", "llama_lora")

# 1. Create a fresh adapter ZIP.
adapter_zip = "/kaggle/working/tinyllama_lora_adapter.zip"

if os.path.isdir(MODEL_DIR):
    shutil.make_archive(
        adapter_zip.removesuffix(".zip"),
        "zip",
        root_dir=MODEL_DIR
    )
    print("Download the TinyLlama adapter:")
    display(FileLink(adapter_zip))
else:
    print("Adapter folder not found:", MODEL_DIR)

# 2. Bundle the predictions and metric files.
results_zip = "/kaggle/working/tinyllama_results_backup.zip"
result_files = [
    "llama_predictions.csv",
    "llama_metrics.json",
    "llama_training_summary.json",
    "llama_inference_summary.json"
]

with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in result_files:
        path = os.path.join(RESULTS_DIR, filename)
        if os.path.isfile(path):
            zf.write(path, arcname=filename)
            print("Included:", filename)
        else:
            print("Missing:", filename)

if zipfile.ZipFile(results_zip).namelist():
    print("\nDownload the TinyLlama results:")
    display(FileLink(results_zip))
else:
    print("No result files found. Check RESULTS_DIR.")


Download the TinyLlama adapter:


/kaggle/working/tinyllama_lora_adapter.zip

Included: llama_predictions.csv
Included: llama_metrics.json
Included: llama_training_summary.json
Included: llama_inference_summary.json

Download the TinyLlama results:


/kaggle/working/tinyllama_results_backup.zip

In [19]:

import os
import shutil
from IPython.display import display, FileLink

# Check the files first
files_to_check = {
    "Adapter": "/kaggle/working/tinyllama_lora_adapter.zip",
    "Results": "/kaggle/working/tinyllama_results_backup.zip",
    "Predictions": "/kaggle/working/ICT4403_MT_Project/results/llama/llama_predictions.csv",
}

for name, path in files_to_check.items():
    if os.path.isfile(path):
        print(f"{name}: FOUND ({os.path.getsize(path) / (1024**2):.2f} MB)")
        display(FileLink(path))
    else:
        print(f"{name}: NOT FOUND — {path}")


Adapter: FOUND (21.38 MB)


/kaggle/working/tinyllama_lora_adapter.zip

Results: FOUND (0.09 MB)


/kaggle/working/tinyllama_results_backup.zip

Predictions: FOUND (0.22 MB)


/kaggle/working/ICT4403_MT_Project/results/llama/llama_predictions.csv

In [20]:

import os
import shutil

files = [
    (
        "/kaggle/working/tinyllama_lora_adapter.zip",
        "tinyllama_lora_adapter.zip"
    ),
    (
        "/kaggle/working/tinyllama_results_backup.zip",
        "tinyllama_results_backup.zip"
    ),
]

for source, filename in files:
    if os.path.isfile(source):
        destination = f"/kaggle/working/{filename}"
        if os.path.abspath(source) != os.path.abspath(destination):
            shutil.copy2(source, destination)
        print(f"READY: {destination}")
        print(f"Size: {os.path.getsize(destination):,} bytes\n")
    else:
        print(f"MISSING: {source}")

print("Open the notebook's right-hand Output/Files panel and locate these files.")


READY: /kaggle/working/tinyllama_lora_adapter.zip
Size: 22,417,349 bytes

READY: /kaggle/working/tinyllama_results_backup.zip
Size: 94,717 bytes

Open the notebook's right-hand Output/Files panel and locate these files.



import os

paths = [
    "/kaggle/working/ICT4403_MT_Project/models/llama_lora",
    "/kaggle/working/ICT4403_MT_Project/results/llama/llama_predictions.csv",
    "/kaggle/working/ICT4403_MT_Project/results/llama/llama_training_summary.json",
    "/kaggle/working/ICT4403_MT_Project/results/llama/llama_inference_summary.json",
]

for path in paths:
    print(("FOUND  " if os.path.exists(path) else "MISSING"), path)


import os

for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if d != ".virtual_documents"]
    for filename in files:
        if any(key in filename.lower() for key in [
            "llama_predictions",
            "llama_metrics",
            "adapter_model",
            "adapter_config"
        ]):
            print(os.path.join(root, filename))